# 🔍 Policy Validation

This notebook validates a trained policy against a dataset by comparing predicted actions to ground truth actions.

**What it does:**
1. Loads a trained policy checkpoint
2. Loads a validation dataset
3. Runs the policy on each sample to get predicted actions
4. Plots ground truth vs predicted actions for each action dimension

This helps you visually assess how well your policy has learned the task.

---
## 1. Configuration

Specify the paths to your checkpoint and dataset.

> **Action Required:** Update `CHECKPOINT_DIR` and `DATASET_DIR` below.

In [ ]:
import pathlib

# --- Paths ---
# TODO: Path to your trained policy checkpoint
CHECKPOINT_DIR = pathlib.Path("/data/models/your_checkpoint_here")

# TODO: Path to the dataset to validate against
# This can be the same dataset you trained on, or a held-out validation set
DATASET_DIR = pathlib.Path("/data/lerobot/your_dataset_here")

# --- Validation Settings ---
# Maximum number of episodes to validate (set to None for all)
MAX_EPISODES = 10

# Output path for the plot (None = display inline only)
OUTPUT_PATH = None  # e.g., pathlib.Path("validation_plot.png")

print(f"Checkpoint: {CHECKPOINT_DIR}")
print(f"Dataset: {DATASET_DIR}")
print(f"Max episodes: {MAX_EPISODES if MAX_EPISODES else 'All'}")

---
## 2. Load Policy and Dataset

In [ ]:
import torch
from lerobot.datasets.lerobot_dataset import LeRobotDataset
from example_policies.robot_deploy.deploy_core.policy_loader import load_policy
from example_policies.utils.action_order import ActionMode

# Select device
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Load policy and processors
print(f"\nLoading policy from {CHECKPOINT_DIR}...")
policy, cfg, preprocessor, postprocessor = load_policy(CHECKPOINT_DIR)
policy.to(device)
policy.eval()

# Detect action mode
action_mode = ActionMode.parse_action_mode(cfg)
is_umi_delta = action_mode == ActionMode.UMI_DELTA_TCP

# For UMI-delta: extract TCP pose indices from observation.state metadata
# These are needed to find the reference frame for converting predictions
# back to absolute TCP space for comparison with ground truth.
tcp_indices = None
if is_umi_delta:
    state_names = cfg.metadata["features"]["observation.state"]["names"]
    def _find_idxs(prefix, count):
        start = state_names.index(prefix)
        return list(range(start, start + count))
    tcp_indices = {
        "pos_l": _find_idxs("tcp_left_pos_x", 3),
        "quat_l": _find_idxs("tcp_left_quat_x", 4),
        "pos_r": _find_idxs("tcp_right_pos_x", 3),
        "quat_r": _find_idxs("tcp_right_quat_x", 4),
    }

n_action_steps = getattr(policy.config, 'n_action_steps', 16)

print("✅ Policy loaded successfully!")
print(f"   action_mode: {action_mode}")
if is_umi_delta:
    print("   ℹ️  UMI-delta model: predictions will be converted to abs TCP for validation")

# Load dataset
print(f"\nLoading dataset from {DATASET_DIR}...")
dataset = LeRobotDataset(
    repo_id=str(DATASET_DIR),
    root=DATASET_DIR,
    video_backend="pyav",  # pyav works on JupyterHub, unlike torchcodec
)
print(f"✅ Dataset loaded: {dataset.meta.total_episodes} episodes, {dataset.meta.total_frames} frames")

---
## 3. Run Validation

Run the policy on the dataset and collect predictions vs ground truth.

In [ ]:
from tqdm.auto import tqdm

def to_device_batch(batch: dict, device: torch.device, non_blocking: bool = True):
    """Move all tensors in a batch to the specified device."""
    out = {}
    for k, v in batch.items():
        if torch.is_tensor(v):
            out[k] = v.to(device, non_blocking=non_blocking)
        else:
            out[k] = v
    return out

def squeeze_temporal_dim(batch: dict) -> dict:
    """Squeeze singleton temporal dim from video features for select_action.
    
    LeRobot datasets return images as [B, T, C, H, W] but select_action
    manages temporal queuing internally, so it expects [B, C, H, W].
    """
    out = {}
    for k, v in batch.items():
        if torch.is_tensor(v) and "image" in k and v.dim() == 5:
            out[k] = v.squeeze(1)
        else:
            out[k] = v
    return out

# ---------- UMI-delta → abs TCP conversion ----------
def _is_new_chunk(policy):
    """Check if the next select_action call will generate a new action chunk."""
    queues = getattr(policy, '_queues', None)
    if queues is None:
        return True
    action_queue = queues.get("action")
    if action_queue is None:
        return True
    return len(action_queue) == 0

if is_umi_delta:
    from example_policies.data_ops.utils.rotation_6d import (
        compose_transform_6d_torch,
        quat_to_6d_torch,
        rotation_6d_to_quat_torch,
    )
    from example_policies.utils.action_order import (
        DUAL_ABS_LEFT_POS_IDXS, DUAL_ABS_LEFT_QUAT_IDXS,
        DUAL_ABS_RIGHT_POS_IDXS, DUAL_ABS_RIGHT_QUAT_IDXS,
        UMI_LEFT_POS_IDXS, UMI_LEFT_ROT6D_IDXS,
        UMI_RIGHT_POS_IDXS, UMI_RIGHT_ROT6D_IDXS,
        UMI_LEFT_GRIPPER_IDX, UMI_RIGHT_GRIPPER_IDX,
        UMI_ACTION_DIM,
    )
    from example_policies.utils.chunk_relative_processor import abs_tcp_to_chunk_relative_umi_delta

def _match_quaternion_sign(pred_quat: torch.Tensor, gt_quat: torch.Tensor) -> torch.Tensor:
    """Flip predicted quaternion sign to match ground truth (closest hemisphere)."""
    gt = gt_quat.to(pred_quat.device)
    if torch.dot(pred_quat, gt) < 0:
        return -pred_quat
    return pred_quat

def umi_delta_to_abs_tcp(umi_action, ref_state):
    """Convert a single UMI-delta action (20-dim) to abs TCP (16-dim)."""
    umi = umi_action.squeeze(0) if umi_action.dim() > 1 else umi_action
    dev = umi.device
    abs_tcp = torch.zeros(16, device=dev, dtype=umi.dtype)

    for side, pos_sl, rot6d_sl, abs_pos_sl, abs_quat_sl, grip_idx in [
        ("l", UMI_LEFT_POS_IDXS, UMI_LEFT_ROT6D_IDXS,
         DUAL_ABS_LEFT_POS_IDXS, DUAL_ABS_LEFT_QUAT_IDXS, UMI_LEFT_GRIPPER_IDX),
        ("r", UMI_RIGHT_POS_IDXS, UMI_RIGHT_ROT6D_IDXS,
         DUAL_ABS_RIGHT_POS_IDXS, DUAL_ABS_RIGHT_QUAT_IDXS, UMI_RIGHT_GRIPPER_IDX),
    ]:
        ref_pos = ref_state[tcp_indices[f"pos_{side}"]].to(dev)
        ref_quat = ref_state[tcp_indices[f"quat_{side}"]].to(dev)
        ref_rot6d = quat_to_6d_torch(ref_quat)

        abs_pos, abs_rot6d = compose_transform_6d_torch(
            ref_pos, ref_rot6d, umi[pos_sl], umi[rot6d_sl]
        )
        abs_quat = rotation_6d_to_quat_torch(abs_rot6d)
        abs_quat = abs_quat / abs_quat.norm()

        abs_tcp[abs_pos_sl] = abs_pos
        abs_tcp[abs_quat_sl] = abs_quat

    abs_tcp[14] = umi[UMI_LEFT_GRIPPER_IDX]
    abs_tcp[15] = umi[UMI_RIGHT_GRIPPER_IDX]
    return abs_tcp

# Create dataloader
dataloader = torch.utils.data.DataLoader(
    dataset,
    num_workers=4,
    batch_size=1,
    shuffle=False,
    pin_memory=device != "cpu",
    drop_last=False,
)

# Dictionary to store data for each episode
episodes_data = {}
action_dim = None

print(f"Running validation on up to {MAX_EPISODES if MAX_EPISODES else 'all'} episodes...")
prev_ep = -1
chunk_ref_state = None  # reference observation.state at chunk boundary (UMI-delta)

# Collect predictions
with torch.no_grad():
    for batch in tqdm(dataloader, desc="Validating"):
        # Get episode index
        b_ep = batch.get("episode_index")
        if b_ep is None:
            raise KeyError("Expected key 'episode_index' in batch.")
        b_ep = int(b_ep.view(-1)[0].item())
        
        # Skip if we've reached max episodes
        if MAX_EPISODES is not None and b_ep >= MAX_EPISODES:
            break

        # Reset policy when starting a new episode
        if b_ep != prev_ep:
            policy.reset()
            prev_ep = b_ep
        
        # Initialize episode data structure if needed
        if b_ep not in episodes_data:
            episodes_data[b_ep] = {
                "targets": [],
                "preds": [],
                "times": [],
                "targets_6d": [],
                "preds_6d": [],
                "chunk_boundary_times": [],
            }
        
        # Move batch to device
        batch = to_device_batch(batch, device, non_blocking=True)
        
        # Get ground truth action (always abs TCP from dataset)
        tgt = batch["action"].detach().float().view(-1)
        if action_dim is None:
            action_dim = tgt.numel()
        
        # Remove action from obs before preprocessing — during deployment
        # the observation dict never contains actions.
        obs = {k: v for k, v in batch.items() if k != "action"}

        # Detect whether a new action chunk will be predicted this step.
        _queues = getattr(policy, "_queues", None)
        is_new_chunk = (
            _queues is None
            or len(_queues.get("action", [])) == 0
        )
        
        # For UMI-delta: capture reference state at chunk boundaries
        if is_umi_delta and is_new_chunk:
            ref = batch["observation.state"].detach().float().squeeze(0)
            if ref.dim() > 1:
                ref = ref[-1]  # take last observation step
            chunk_ref_state = ref
            t_boundary = float(batch["timestamp"].view(-1)[0].detach().cpu().item())
            episodes_data[b_ep]["chunk_boundary_times"].append(t_boundary)
        
        # Apply preprocessor if available (normalization)
        if preprocessor is not None:
            obs = preprocessor(obs)
        
        # Squeeze singleton temporal dim from images for select_action
        obs = squeeze_temporal_dim(obs)
        
        # Get predicted action
        action = policy.select_action(obs)
        
        # Apply postprocessor if available (unnormalization)
        # For stepwise unnormalizers we must unnormalize the full chunk at once
        # so that each timestep gets its correct per-step stats (p02[k], p98[k]).
        if postprocessor is not None:
            if is_new_chunk and _queues is not None:
                queue = policy._queues["action"]
                remaining = list(queue)
                full_chunk = torch.stack([action] + remaining, dim=1)
                full_chunk = postprocessor(full_chunk)
                action = full_chunk[:, 0]
                queue.clear()
                for t in range(1, full_chunk.shape[1]):
                    queue.append(full_chunk[:, t])
            elif not is_new_chunk and _queues is not None:
                # Already unnormalized when the chunk was processed above.
                pass
            else:
                # No queue (e.g. ACT) — unnormalize individually.
                action = postprocessor(action)
        
        pred = action.detach().float().view(-1)
        
        # For UMI-delta: convert prediction from 20-dim UMI-delta to
        # 16-dim abs TCP so it can be compared against the ground truth.
        if is_umi_delta and chunk_ref_state is not None:
            # Keep the raw UMI-delta (6D) for the second plot
            pred_6d = pred.clone().cpu()

            # Convert GT abs TCP to UMI-delta for direct 6D comparison
            ref = chunk_ref_state
            ref_pos_l = ref[tcp_indices["pos_l"]]
            ref_quat_l = ref[tcp_indices["quat_l"]]
            ref_rot6d_l = quat_to_6d_torch(ref_quat_l)
            ref_pos_r = ref[tcp_indices["pos_r"]]
            ref_quat_r = ref[tcp_indices["quat_r"]]
            ref_rot6d_r = quat_to_6d_torch(ref_quat_r)
            tgt_16 = tgt.unsqueeze(0).unsqueeze(0)  # (1,1,16)
            tgt_6d = abs_tcp_to_chunk_relative_umi_delta(
                tgt_16, ref_pos_l, ref_rot6d_l, ref_pos_r, ref_rot6d_r,
            ).squeeze(0).squeeze(0).cpu()  # (20,)

            episodes_data[b_ep]["targets_6d"].append(tgt_6d)
            episodes_data[b_ep]["preds_6d"].append(pred_6d)

            # Convert prediction to abs TCP for the main comparison plot
            pred = umi_delta_to_abs_tcp(pred, chunk_ref_state)
            # Fix quaternion sign ambiguity
            pred[DUAL_ABS_LEFT_QUAT_IDXS] = _match_quaternion_sign(
                pred[DUAL_ABS_LEFT_QUAT_IDXS], tgt[DUAL_ABS_LEFT_QUAT_IDXS]
            )
            pred[DUAL_ABS_RIGHT_QUAT_IDXS] = _match_quaternion_sign(
                pred[DUAL_ABS_RIGHT_QUAT_IDXS], tgt[DUAL_ABS_RIGHT_QUAT_IDXS]
            )
        
        # Store data (move everything to CPU for storage)
        episodes_data[b_ep]["targets"].append(tgt.cpu())
        episodes_data[b_ep]["preds"].append(pred.cpu())
        
        # Store timestamp
        t = float(batch["timestamp"].view(-1)[0].detach().cpu().item())
        episodes_data[b_ep]["times"].append(t)

# Stack data for each episode
for ep_idx in episodes_data:
    episodes_data[ep_idx]["targets"] = torch.stack(episodes_data[ep_idx]["targets"], dim=0)
    episodes_data[ep_idx]["preds"] = torch.stack(episodes_data[ep_idx]["preds"], dim=0)
    episodes_data[ep_idx]["times"] = torch.tensor(episodes_data[ep_idx]["times"])
    if episodes_data[ep_idx]["targets_6d"]:
        episodes_data[ep_idx]["targets_6d"] = torch.stack(episodes_data[ep_idx]["targets_6d"], dim=0)
        episodes_data[ep_idx]["preds_6d"] = torch.stack(episodes_data[ep_idx]["preds_6d"], dim=0)

num_episodes = len(episodes_data)

# Get action names from model metadata for labelling
action_names = cfg.metadata.get("features", {}).get("action", {}).get("names", None)

print(f"\n✅ Validated {num_episodes} episodes")

---
## 4. Compute Metrics

Calculate error metrics to quantify policy performance.

In [ ]:
import numpy as np

# Concatenate all episode data
all_targets = torch.cat([episodes_data[ep]["targets"] for ep in sorted(episodes_data.keys())], dim=0)
all_preds = torch.cat([episodes_data[ep]["preds"] for ep in sorted(episodes_data.keys())], dim=0)
D_act = all_targets.shape[1]

print(f"Per-dimension MAE — abs TCP ({num_episodes} episodes, {all_targets.shape[0]} frames):")
print("=" * 70)
print(f"{'dim':>4}  {'label':>20}  {'MAE':>10}  {'GT_range':>10}  {'MAE/range':>10}")
print("-" * 70)

for d in range(D_act):
    label = action_names[d] if action_names and d < len(action_names) else f"dim_{d}"
    mae = (all_targets[:, d] - all_preds[:, d]).abs().mean().item()
    gt_range = all_targets[:, d].max().item() - all_targets[:, d].min().item()
    ratio = mae / gt_range if gt_range > 1e-8 else float('inf')
    print(f"  [{d:2d}] {label:>20}  {mae:>10.6f}  {gt_range:>10.6f}  {ratio:>10.2%}")

print("=" * 70)
overall_mae = (all_targets - all_preds).abs().mean().item()
print(f"Overall MAE: {overall_mae:.6f}")

# --- 6D UMI-delta metrics ---
if is_umi_delta:
    umi_dim_names = [
        "L_dx", "L_dy", "L_dz",
        "L_r6d_0", "L_r6d_1", "L_r6d_2", "L_r6d_3", "L_r6d_4", "L_r6d_5",
        "R_dx", "R_dy", "R_dz",
        "R_r6d_0", "R_r6d_1", "R_r6d_2", "R_r6d_3", "R_r6d_4", "R_r6d_5",
        "grip_L", "grip_R",
    ]
    all_t6 = torch.cat([episodes_data[e]["targets_6d"] for e in sorted(episodes_data)], dim=0)
    all_p6 = torch.cat([episodes_data[e]["preds_6d"] for e in sorted(episodes_data)], dim=0)
    D6 = all_t6.shape[1]

    print(f"\nPer-dimension MAE — 6D UMI-delta space ({num_episodes} episodes, {all_t6.shape[0]} frames):")
    print("=" * 70)
    print(f"{'dim':>4}  {'label':>12}  {'MAE':>10}  {'GT_range':>10}  {'MAE/range':>10}")
    print("-" * 70)
    for d in range(D6):
        mae = (all_t6[:, d] - all_p6[:, d]).abs().mean().item()
        gt_range = all_t6[:, d].max().item() - all_t6[:, d].min().item()
        ratio = mae / gt_range if gt_range > 1e-8 else float('inf')
        print(f"  [{d:2d}] {umi_dim_names[d]:>12}  {mae:>10.6f}  {gt_range:>10.6f}  {ratio:>10.2%}")
    print("=" * 70)

---
## 5. Plot Results

Visualize ground truth vs predicted actions for each action dimension.

In [ ]:
import matplotlib.pyplot as plt

if num_episodes == 0:
    print("No episodes found in dataset.")
else:
    # Get action dimension from first episode
    first_ep = list(episodes_data.keys())[0]
    D = episodes_data[first_ep]["targets"].shape[1]
    
    # Create plots
    fig, axes = plt.subplots(D, 1, figsize=(12, 2.2 * D), sharex=True)
    if D == 1:
        axes = [axes]
    
    # Plot each episode
    for ep_idx in sorted(episodes_data.keys()):
        ep_data = episodes_data[ep_idx]
        times = ep_data["times"].numpy()
        targets = ep_data["targets"].numpy()
        preds = ep_data["preds"].numpy()
        
        color = f"C{ep_idx % 10}"
        
        for d in range(D):
            ax = axes[d]
            ax.plot(times, targets[:, d], color=color, alpha=0.7, 
                    label=f"Ep {ep_idx} GT", linestyle='-')
            ax.plot(times, preds[:, d], color=color, alpha=0.7, 
                    label=f"Ep {ep_idx} Pred", linestyle='--')
            dim_label = action_names[d] if action_names and d < len(action_names) else f"dim {d}"
            ax.set_ylabel(dim_label, fontsize=7)
            ax.grid(True, linestyle="--", alpha=0.3)
            if d == 0:
                title = f"Abs TCP: GT vs predictions ({num_episodes} episodes)"
                if is_umi_delta:
                    title += "  (UMI-delta → abs TCP)"
                ax.set_title(title)
    
    axes[-1].set_xlabel("Time (s)")
    
    if num_episodes <= 3:
        axes[0].legend(loc="upper right", fontsize='small')
    else:
        handles, labels = axes[0].get_legend_handles_labels()
        fig.legend(handles[:6], labels[:6], loc="upper right", fontsize='small')
    
    plt.tight_layout(rect=[0, 0, 0.98, 0.98])
    
    if OUTPUT_PATH:
        fig.savefig(OUTPUT_PATH, dpi=150, bbox_inches='tight')
        print(f"\n💾 Saved abs TCP plot to {OUTPUT_PATH}")
    
    plt.show()

---
## 5b. UMI-Delta 6D Plot (UMI-delta models only)

For UMI-delta models this second plot shows ground truth vs predictions directly in the 20-dim chunk-relative UMI-delta space (position deltas + 6D rotation), avoiding quaternion conversion artefacts. Dotted vertical lines mark chunk boundaries.

In [ ]:
if is_umi_delta and num_episodes > 0:
    umi_dim_names = [
        "L_dx", "L_dy", "L_dz",
        "L_r6d_0", "L_r6d_1", "L_r6d_2", "L_r6d_3", "L_r6d_4", "L_r6d_5",
        "R_dx", "R_dy", "R_dz",
        "R_r6d_0", "R_r6d_1", "R_r6d_2", "R_r6d_3", "R_r6d_4", "R_r6d_5",
        "grip_L", "grip_R",
    ]
    D6 = UMI_ACTION_DIM

    fig6, axes6 = plt.subplots(D6, 1, figsize=(12, 2.2 * D6), sharex=True)
    if D6 == 1:
        axes6 = [axes6]

    for ep_idx in sorted(episodes_data.keys()):
        ep_data = episodes_data[ep_idx]
        times = ep_data["times"].numpy()
        t6 = ep_data["targets_6d"].numpy()
        p6 = ep_data["preds_6d"].numpy()
        color = f"C{ep_idx % 10}"
        chunk_times = ep_data["chunk_boundary_times"]

        for d in range(D6):
            ax = axes6[d]
            ax.plot(times, t6[:, d], color=color, alpha=0.7, linestyle='-',
                    label=f"Ep {ep_idx} GT" if d == 0 else None)
            ax.plot(times, p6[:, d], color=color, alpha=0.7, linestyle='--',
                    label=f"Ep {ep_idx} Pred" if d == 0 else None)
            # Mark chunk boundaries with thin vertical lines
            for ct in chunk_times:
                ax.axvline(ct, color='gray', alpha=0.3, linewidth=0.5, linestyle=':')
            ax.set_ylabel(umi_dim_names[d], fontsize=7)
            ax.grid(True, linestyle="--", alpha=0.3)
            if d == 0:
                ax.set_title(f"6D UMI-delta space ({num_episodes} episodes): GT vs predictions  (dotted = chunk boundaries)")

    axes6[-1].set_xlabel("Time (s)")
    if num_episodes <= 3:
        axes6[0].legend(loc="upper right", fontsize='small')
    else:
        handles, labels = axes6[0].get_legend_handles_labels()
        fig6.legend(handles[:6], labels[:6], loc="upper right", fontsize='small')

    plt.tight_layout(rect=[0, 0, 0.98, 0.98])

    if OUTPUT_PATH:
        output_6d = OUTPUT_PATH.parent / (OUTPUT_PATH.stem + "_6d" + OUTPUT_PATH.suffix)
        fig6.savefig(output_6d, dpi=150, bbox_inches='tight')
        print(f"\n💾 Saved 6D UMI-delta plot to {output_6d}")

    plt.show()
else:
    print("ℹ️  Not a UMI-delta model — skipping 6D plot.")

---
## 6. Per-Dimension Error Analysis (Optional)

Detailed error statistics and histograms for each action dimension.

In [ ]:
# Detailed per-dimension error stats (bias, std, max)
errors = all_targets.numpy() - all_preds.numpy()

print("Per-Dimension Error Statistics (abs TCP):")
print("=" * 80)
print(f"{'dim':>4}  {'label':>20}  {'Mean Err':>12}  {'Std Err':>12}  {'MAE':>12}  {'Max|Err|':>12}")
print("-" * 80)

for d in range(errors.shape[1]):
    label = action_names[d] if action_names and d < len(action_names) else f"dim_{d}"
    dim_errors = errors[:, d]
    print(f"  [{d:2d}] {label:>20}  {np.mean(dim_errors):>12.6f}  {np.std(dim_errors):>12.6f}  "
          f"{np.mean(np.abs(dim_errors)):>12.6f}  {np.max(np.abs(dim_errors)):>12.6f}")

print("=" * 80)

In [ ]:
# Error distribution histograms (grid layout for many dimensions)
D = errors.shape[1]
ncols = min(D, 4)
nrows = (D + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 3 * nrows))
axes = np.atleast_2d(axes)

for d in range(D):
    r, c = divmod(d, ncols)
    ax = axes[r, c]
    ax.hist(errors[:, d], bins=50, edgecolor='black', alpha=0.7)
    ax.axvline(x=0, color='red', linestyle='--', linewidth=1)
    ax.set_xlabel('Error')
    ax.set_ylabel('Count')
    label = action_names[d] if action_names and d < len(action_names) else f"dim {d}"
    ax.set_title(label, fontsize=9)
    ax.grid(True, alpha=0.3)

# Hide unused subplots
for d in range(D, nrows * ncols):
    r, c = divmod(d, ncols)
    axes[r, c].set_visible(False)

plt.suptitle('Error Distribution per Action Dimension (abs TCP)', fontsize=14)
plt.tight_layout()
plt.show()

---
## ✅ Done!

**Interpreting Results:**

- **Low MSE/MAE**: The policy accurately predicts the demonstrated actions
- **High MSE/MAE**: The policy may need more training or the task is challenging
- **Bias in error**: If mean error ≠ 0, the policy has a systematic bias
- **High variance in specific dimensions**: Some action components may be harder to learn

**Note:** Good validation metrics don't guarantee good real-world performance. Always test on the actual robot!